### <span style="color:blue">Checking elasticsearch connection and paths of data</span>

In [ ]:
# check if the elasticsearch container is running
from elasticsearch import Elasticsearch

!curl http://localhost:9200/

In [ ]:
# Check paths
import os
print(os.getcwd())

print(os.path.exists("../data/documents.csv"))

### <span style="color:blue">Define elasticsearch client and paths of data</span>

In [ ]:
# instantiate Elasticsearch client localhost
client = Elasticsearch("http://localhost:9200")

In [ ]:
# Define paths
from pathlib import Path

BASE_DIR = Path().resolve().parent  # project
doc_csv_path = BASE_DIR / "data" / "documents.csv"
doc_jsonl_path = BASE_DIR / "data" / "documents.jsonl"

queries_csv_path = BASE_DIR / "data" / "queries.csv"
queries_jsonl_path = BASE_DIR / "data" / "queries.jsonl"


### <span style="color:blue">Definition of functions</span>

In [ ]:
# csv to jsonl file conversion function
import csv
import json

def csv_to_jsonl(csv_path, jsonl_path, encoding="utf-8"):
    with open(csv_path, mode="r", encoding=encoding, newline="") as csv_file:
        reader = csv.DictReader(csv_file)

        with open(jsonl_path, mode="w", encoding=encoding) as jsonl_file:
            for row in reader:
                json_line = json.dumps(row, ensure_ascii=False)
                jsonl_file.write(json_line + "\n")

In [ ]:
# jsonl file reading function
def read_json_data(jsonl_path):
    all_json_data = [] # a list of json records
    with open(jsonl_path, "r", encoding="utf-8") as file:
        for line in file:
            json_data = json.loads(line)
            all_json_data.append(json_data)
    return all_json_data

In [ ]:
# Index creation function
# Use bulk helpers
from elasticsearch.helpers import bulk

def create_index(json_data_list, index_name):
    documents = []

    for doc in json_data_list:
        documents.append({
            "_index": index_name,
            "_source": doc
        })

    success, errors = bulk(
        client,
        documents,
        refresh=True,
        raise_on_error=False   
    )

    print("Bulk success:", success)
    print("Bulk errors:", len(errors))

    if errors:
        print("First error example:")
        print(errors[0])

    print(f"Done indexing documents into {index_name} index!")


In [ ]:
# Function to delete the index anytime
def delete_index(index_name):
    if client.indices.exists(index=index_name):
        client.indices.delete(index=index_name)
        print(f"Index '{index_name}' deleted.")
    else:
        print(f"Index '{index_name}' does not exists.")

In [ ]:
# Function that returns a list of answers

def search_response(response, max_docs=200):
    results = []

    hits = response.get("hits", {}).get("hits", [])

    if len(hits) == 0:
        return results

    for hit in hits[:max_docs]:
        results.append({
            "doc_id": hit["_source"]["ID"],
            "text": hit["_source"]["Text"],
            "bm25_score": hit["_score"]
        })

    return results


In [ ]:
# Preprocessing function cleans up "Text

from cleantext import clean

def preprocess(batch):
    texts = batch["Text"] # batch στην dataset.map()
    cleaned = []

    for text in texts:
        # Basic cleaning uses the clean function from the clean-text package pypi.org
        t = clean(
            text,
            fix_unicode=True,           # δfixes strange characters
            to_ascii=False,             # holds non-ASCII (e.g. names)
            lower=False,                # no lowercase for BERT-like models
            no_urls=True,
            no_emails=True,
            no_phone_numbers=True,
            no_line_breaks=True,
            no_punct=False,             # we keep punctuation marks (the tokenizer needs them)
            no_digits=False,            
            no_currency_symbols=False
        )

        # Light normalization
        t = t.strip()                   # removing unnecessary spaces
        t = " ".join(t.split())         # normalize whitespace

        cleaned.append(t)

    batch["clean_text"] = cleaned
    return batch

### <span style="color:blue">Convert csv files to jsonl</span>

In [ ]:
# Convert documents.csv file to documents.jsonl
csv_to_jsonl(doc_csv_path, doc_jsonl_path)

In [ ]:
# Convert queries.csv file to queries.jsonl
csv_to_jsonl(queries_csv_path, queries_jsonl_path)

In [ ]:
# List of data in documents.jsonl
all_json_data = read_json_data(doc_jsonl_path)

In [ ]:
# List of queries in queries.jsonl
all_json_queries = read_json_data(queries_jsonl_path)

In [ ]:
# testing lists of data and queries
print(f"type of 'all_json_data': {type(all_json_data)}\nfirst row of data: {all_json_data[0]}\nsize of data: {len(all_json_data)}\n")
print(f"type of 'all_json_queries': {type(all_json_queries)}\nfirst row of queries: {all_json_queries[0]}\nsize of queries: {len(all_json_queries)}")


### <span style="color:blue">Mapping and create index BM25</span>

In [ ]:
# Call delete_index(index_name) any time - check before call create_index function
# if exists endpoint: http://localhost:9200/my_data
index_name = "my_data"
delete_index(index_name)

In [ ]:
# configure index properties
# create fields
mapping = {
  "settings": {
    "number_of_shards": 1,
    "index": {
      "similarity": {
        "default": {
          "type": "BM25",
          "b": 0.8,
          "k1": 1.8
        }
      }
    },
    "analysis": {
      "analyzer": {
        "default": {
          "type": "english"
        },
        "default_search": {
          "type": "english"
        }
      }
    }
  },
  "mappings": {
    "properties": {
      "id": {
        "type": "keyword"
      },
      "text": {
        "type": "text"
      }
    }
  }
}

# create an index with the configuration above
client.indices.create(index='my_data', body=mapping)

In [ ]:
# view mapping
client.indices.get_mapping(index="my_data")

In [ ]:
# Call create_index(json_data_list, index_name)
my_data = create_index(all_json_data, "my_data")

### <span style="color:blue">Download model</span>

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

### <span style="color:blue">Pipeline reranking general idea</span>
```
for every query
    BM25 -> top 200
    preprocess docs + query
    create embeddings
    create faiss index
    for k = 20, 30, 50
        top k
        write in run file
```

### <span style="color:blue">Reranking with faiss</span>
1. Faiss only - create run_faiss files for trec_eval
2. Faiss and BM25 scores - create run_fusion files for trec_eval

In [ ]:
import numpy as np

# function that converts all scores to space [0, 1]
def minmax(scores):
    s = np.array(scores, dtype=float)
    if s.max() == s.min():
        return np.ones_like(s)
    return (s - s.min()) / (s.max() - s.min())

In [ ]:
import json
import faiss

# dictionary run_files - faiss only
run_files = {
    20: open("run_faiss_k20.txt", "w"),
    30: open("run_faiss_k30.txt", "w"),
    50: open("run_faiss_k50.txt", "w"),
}
# dictionary fusion_files - normalize faiss and BM25 scores
fusion_files = {
    20: open("run_fusion_k20.txt", "w"),
    30: open("run_fusion_k30.txt", "w"),
    50: open("run_fusion_k50.txt", "w"),
}

with open(queries_jsonl_path, "r", encoding="utf-8") as f:
    queries = [json.loads(line) for line in f]

    for query in queries:
        query_id = query["ID"]
        query_text_raw = query["Text"]

        # Phase 1 BM25 - elasticsearch response for top 200
        # search with inverted index BM25
        es_response = client.search(
            index="my_data",
            size=200,
            query={"match": {"Text": query_text_raw}}
        )

        results_BM25 = search_response(es_response)  # list of dictionaries

        if len(results_BM25) == 0:
            continue

        # extract texts from results_BM25
        doc_texts = [d["text"] for d in results_BM25]

        # preprocess - clean
        batch = {"Text": [doc["text"] for doc in results_BM25]}
        batch = preprocess(batch)
        clean_data = batch["clean_text"]

        query_batch = {"Text": [query_text_raw]}
        query_batch = preprocess(query_batch)
        clean_query = query_batch["clean_text"]

        # create embeddings
        data_embeddings = model.encode(
            clean_data,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        query_embedding = model.encode(
            clean_query,
            convert_to_numpy=True,
            normalize_embeddings=True
        ).reshape(1, -1)

        # create faiss index
        dim = data_embeddings.shape[1]
        index = faiss.IndexFlatIP(dim)
        index.add(data_embeddings) # faiss index

        # ----------fussion fase-----------------
        # top 200
        scores, indices = index.search(query_embedding, 200)
        
        # Fusion preparation 
        alpha = 0.5  

        bm25_scores = [d["bm25_score"] for d in results_BM25]
        bm25_norm = minmax(bm25_scores)
        faiss_norm = minmax(scores[0])

        fusion_results = []

        for i, doc_idx in enumerate(indices[0]):
            fusion_score = (
                alpha * bm25_norm[doc_idx]
                + (1 - alpha) * faiss_norm[i]
            )

            fusion_results.append({
                "doc_id": results_BM25[doc_idx]["doc_id"],
                "score": fusion_score
            })

        # sort descending order
        fusion_results.sort(key=lambda x: x["score"], reverse=True)

        # ----------fussion fase end-------------

        # run_files - faiss only
        for k in [20, 30, 50]:
            scores, indices = index.search(query_embedding, k)

            for rank, idx in enumerate(indices[0], start=1):
                doc_id = results_BM25[idx]["doc_id"]
                score = float(scores[0][rank - 1])

                run_files[k].write(
                    f"{query_id} Q0 {doc_id} {rank} {score} faiss_only\n"
                )
                
        # fusion_files - taking into account BM25 & faiss scores 
        for k in [20, 30, 50]:
            for rank, item in enumerate(fusion_results[:k], start=1):
                fusion_files[k].write(
                    f"{query_id} Q0 {item['doc_id']} {rank} {item['score']} fusion_bm25_faiss\n"
                )
# close run_files
for f in run_files.values():
    f.close()

# close fusion_files
for f in fusion_files.values():
    f.close()